**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
# Step 0 - import libraries needed for this lab
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# Standard seed used throughout the notebook, for reproducibility
SEED = 42

In [2]:
# 0.1 Load the dataset
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


### Recreating the feature engineering, scaling and selection from before

Same pipeline as the Feature Engineering / Ensemble labs, so results are directly comparable:
1. Drop rows with missing values.
2. Reduce `Cabin` to just the deck letter.
3. Drop `PassengerId` and `Name`.
4. One-hot encode remaining categorical columns.
5. Standard-scale all features.
6. Keep the top 10 features via `SelectKBest` (ANOVA F-test).

In [3]:
# 0.2 Same feature engineering as before ---
spaceship = spaceship.dropna()
spaceship["Cabin"] = spaceship["Cabin"].apply(lambda x: x.split("/")[0])
spaceship = spaceship.drop(columns=["PassengerId", "Name"])
spaceship = pd.get_dummies(spaceship, drop_first=True)

X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

# --- Scaling ---
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

# --- Feature selection ---
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X_scaled, y)
selected_columns = X.columns[selector.get_support()]
X_selected = pd.DataFrame(X_selected, columns=selected_columns, index=X.index)

print("Selected features:", list(selected_columns))
X_selected.head()

Selected features: ['RoomService', 'Spa', 'VRDeck', 'HomePlanet_Europa', 'CryoSleep_True', 'Cabin_B', 'Cabin_C', 'Cabin_E', 'Cabin_F', 'Destination_TRAPPIST-1e']


,RoomService,Spa,VRDeck,HomePlanet_Europa,CryoSleep_True,Cabin_B,Cabin_C,Cabin_E,Cabin_F,Destination_TRAPPIST-1e
0,-0.345756,-0.273759,-0.269534,1.717147,-0.738664,3.085305,-0.312289,-0.339578,-0.695098,0.666047
1,-0.176748,0.206165,-0.230494,-0.582361,-0.738664,-0.324117,-0.312289,-0.339578,1.438646,0.666047
2,-0.279083,5.596357,-0.226058,1.717147,-0.738664,-0.324117,-0.312289,-0.339578,-0.695098,0.666047
3,-0.345756,2.636384,-0.098291,1.717147,-0.738664,-0.324117,-0.312289,-0.339578,-0.695098,0.666047
4,0.124056,0.220152,-0.267759,-0.582361,-0.738664,-0.324117,-0.312289,-0.339578,1.438646,0.666047


In [4]:
# step 1 Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=SEED
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (5284, 10)
X_test shape: (1322, 10)


In [5]:
# 1.1 Reusable evaluation helper
def evaluate(model, X_test, y_test, label="Model"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"--- {label} ---")
    print("Accuracy:", acc)
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    return acc

# STEP 2 Use the best model
- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

Across the previous Ensemble Methods lab, **Gradient Boosting** came out on top (~78.4% accuracy), so that's the model we'll fine-tune here. First, let's re-fit it with its **default hyperparameters** as our "before tuning" baseline.

In [6]:
# 2.1 Gradient Boosting Classifier
baseline_gb = GradientBoostingClassifier(random_state=SEED)
baseline_gb.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [7]:
# 2.2 - Evaluate your model
baseline_acc = evaluate(baseline_gb, X_test, y_test, "Gradient Boosting (default hyperparameters)")

--- Gradient Boosting (default hyperparameters) ---
Accuracy: 0.783661119515885

Classification Report:
               precision    recall  f1-score   support

       False       0.83      0.71      0.76       653
        True       0.75      0.85      0.80       669

    accuracy                           0.78      1322
   macro avg       0.79      0.78      0.78      1322
weighted avg       0.79      0.78      0.78      1322

Confusion Matrix:
 [[465 188]
 [ 98 571]]


**STEP 3. Grid/Random Search**
For this lab we will use Grid Search.
We'll search over the hyperparameters that matter most for Gradient Boosting:
- **`n_estimators`**: number of boosting stages (trees).
- **`learning_rate`**: how much each tree corrects the previous ensemble (smaller = more conservative, usually needs more trees).
- **`max_depth`**: depth of each individual tree (controls complexity/overfitting).
- **`subsample`**: fraction of training rows used per tree (< 1.0 adds regularization, like stochastic gradient boosting).

In [8]:
# 3.1. Define hyperparameters to fine tune.
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0]
}

# 3.2 
`GridSearchCV` exhaustively tries every combination in `param_grid` with 5-fold cross-validation, scoring by accuracy, and keeps the best combination. `3 x 3 x 3 x 2 = 54` combinations x 5 folds = 270 fits — `n_jobs=-1` parallelizes across CPU cores to keep this fast.

In [ ]:
# 3.2 - Run Grid Search
grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=SEED),
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


In [ ]:
# 3.3 - Evaluate your model
best_gb = grid_search.best_estimator_
tuned_acc = evaluate(best_gb, X_test, y_test, "Gradient Boosting (tuned)")

In [ ]:
print(f"Default hyperparameters accuracy: {baseline_acc:.4f}")
print(f"Tuned hyperparameters accuracy:   {tuned_acc:.4f}")
print(f"Improvement: {(tuned_acc - baseline_acc) * 100:.2f} percentage points")